# Baseline and no_cons on the same graph

In [5]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

def get_csv_files(case_study):
    """
    Select the CSV file dictionary based on the case study name.
    
    Parameters:
        case_study (str): The name of the case study for selecting the correct CSV files.
    
    Returns:
        dict: A dictionary with scenario names as keys and lists of CSV file paths as values.
    """
    csv_folder = f'./plots/{case_study}'  # Define the folder where CSV files are located
    
    # Define dictionaries for each case study
    case_study_csv_files = {
        "equitysilver": {
            "Baseline": [
                os.path.join(csv_folder, "equitysilver_max_hv_bau_eqtslvr_compile_scenario.csv"),
                os.path.join(csv_folder, "equitysilver_max_st_bau_eqtslvr_compile_scenario_maxstock.csv"),
                os.path.join(csv_folder, "equitysilver_min_em_bau_eqtslvr_compile_scenario_minemission.csv"),
                os.path.join(csv_folder, "equitysilver_min_ha_bau_eqtslvr_compile_scenario.csv")
            ],
            "S0": [
                os.path.join(csv_folder, "equitysilver_max_hv_evenflow_cons_compile_scenario.csv"),
                os.path.join(csv_folder, "equitysilver_max_st_evenflow_cons_compile_scenario_maxstock.csv"),
                os.path.join(csv_folder, "equitysilver_min_em_evenflow_cons_compile_scenario_minemission.csv"),
                os.path.join(csv_folder, "equitysilver_min_ha_evenflow_cons_compile_scenario.csv")
            ]
        },
        "goldenbear": {
                    "Baseline": [
                        os.path.join(csv_folder, "goldenbear_max_hv_bau_gldbr_compile_scenario.csv"),
                        os.path.join(csv_folder, "goldenbear_max_st_bau_gldbr_compile_scenario_maxstock.csv"),
                        os.path.join(csv_folder, "goldenbear_min_em_bau_gldbr_compile_scenario_minemission.csv"),
                        os.path.join(csv_folder, "goldenbear_min_ha_bau_gldbr_compile_scenario.csv")
                    ],
                    "S0": [
                        os.path.join(csv_folder, "goldenbear_max_hv_evenflow_cons_compile_scenario.csv"),
                        os.path.join(csv_folder, "goldenbear_max_st_evenflow_cons_compile_scenario_maxstock.csv"),
                        os.path.join(csv_folder, "goldenbear_min_em_evenflow_cons_compile_scenario_minemission.csv"),
                        os.path.join(csv_folder, "goldenbear_min_ha_evenflow_cons_compile_scenario.csv")
                    ]
                },
        "redchris": {
            "Baseline": [
                os.path.join(csv_folder, "redchris_max_hv_bau_redchrs_compile_scenario.csv"),
                os.path.join(csv_folder, "redchris_max_st_bau_redchrs_compile_scenario_maxstock.csv"),
                os.path.join(csv_folder, "redchris_min_em_bau_redchrs_compile_scenario_minemission.csv"),
                os.path.join(csv_folder, "redchris_min_ha_bau_redchrs_compile_scenario.csv")
            ],
            "S0": [
                os.path.join(csv_folder, "redchris_max_hv_evenflow_cons_compile_scenario.csv"),
                os.path.join(csv_folder, "redchris_max_st_evenflow_cons_compile_scenario_maxstock.csv"),
                os.path.join(csv_folder, "redchris_min_em_evenflow_cons_compile_scenario_minemission.csv"),
                os.path.join(csv_folder, "redchris_min_ha_evenflow_cons_compile_scenario.csv")
            ]
        },
    }
    
    # Return the CSV files for the selected case study
    return case_study_csv_files.get(case_study, {})

def plot_combined_scenarios(csv_files_dict, obj_mode, scenario_names, case_study):
    """
    Plot data from multiple scenarios into a single figure with shared Y-axis ranges for columns.
    
    Parameters:
        csv_files_dict (dict): Dictionary where keys are scenario names and values are lists of CSV file paths.
        case_study (str): Case study name for output file naming.
        obj_mode (list of str): List of objective modes corresponding to the CSV files.
        scenario_names (list of str): List of scenario names for labeling and output file naming.
    """
    # Initialize figure and axes for subplots
    num_scenarios = len(scenario_names)
    fig, axes = plt.subplots(len(obj_mode), 3, figsize=(12, 4 * len(obj_mode)))
    
    # Shared y-axis ranges
    all_oha, all_ohv, all_ogs = [], [], []
    dfs_dict = {scenario: [pd.read_csv(file) for file in files] for scenario, files in csv_files_dict.items()}
    
    for scenario_dfs in dfs_dict.values():
        for df in scenario_dfs:
            all_oha.extend(df.oha)
            all_ohv.extend(df.ohv)
            all_ogs.extend(df.ogs)
    
    common_ylim_oha = (0, max(all_oha) * 1.1)
    common_ylim_ohv = (0, max(all_ohv) * 1.1)
    common_ylim_ogs = (0, max(all_ogs) * 1.1)
    
    # Plot each objective mode for each scenario
    for i, obj in enumerate(obj_mode):
        for j, scenario in enumerate(scenario_names):
            df = dfs_dict[scenario][i]
            # Plot harvested area (ha)
            axes[i, 0].bar(df.period + j * 0.2, df.oha, width=0.2, label=scenario if i == 0 else "")
            axes[i, 0].set_ylim(common_ylim_oha)
            axes[i, 0].set_title(f'{obj}: Harvested Area (ha)')
            
            # Plot harvested volume (m3)
            axes[i, 1].bar(df.period + j * 0.2, df.ohv, width=0.2, label=scenario if i == 0 else "")
            axes[i, 1].set_ylim(common_ylim_ohv)
            axes[i, 1].set_title(f'{obj}: Harvested Volume (m3)')
            
            # Plot growing stock (m3)
            axes[i, 2].bar(df.period + j * 0.2, df.ogs, width=0.2, label=scenario if i == 0 else "")
            axes[i, 2].set_ylim(common_ylim_ogs)
            axes[i, 2].set_title(f'{obj}: Growing Stock (m3)')
        
        # Add axis labels
        axes[i, 0].set_ylabel("Value")
        if i == len(obj_mode) - 1:
            axes[i, 0].set_xlabel("Period")
            axes[i, 1].set_xlabel("Period")
            axes[i, 2].set_xlabel("Period")
    
    # Add legend
    axes[0, 0].legend(title="Scenario", loc='upper right', bbox_to_anchor=(1.2, 1))
    
    # Adjust layout
    plt.tight_layout()
    
    # Save the figure
    folder_path = os.path.join('./plots/fig', case_study)
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
    file_name = f"{case_study}_combined_scenarios.pdf"
    file_path = os.path.join(folder_path, file_name)
    plt.savefig(file_path, format='pdf')
    plt.show()
    plt.close()
    print(f"Plot saved to {file_path}")

# Define objective modes and scenario names
obj_mode = ["Max_hv", "Max_st", "Min_em", "Min_ha"]
scenario_names = ["Baseline", "S0"]

# Create a dropdown widget to select case study
case_study_dropdown = widgets.Dropdown(
    options=["equitysilver", 'goldenbear', 'redchris'],  # You can add other case studies here
    value="equitysilver",
    description="Case Study:",
    disabled=False
)

# Create an interactive plot function
def interactive_plot(case_study):
    # Get the CSV files for the selected case study
    csv_files_dict = get_csv_files(case_study)
    
    # Run the plotting function with the selected case study
    plot_combined_scenarios(csv_files_dict, obj_mode, scenario_names, case_study)

# Link the widget to the plot function
interact(interactive_plot, case_study=case_study_dropdown)


interactive(children=(Dropdown(description='Case Study:', options=('equitysilver', 'goldenbear', 'redchris'), …

<function __main__.interactive_plot(case_study)>